# 🧬 Self-Replicating Agent — Kaggle GPU + Multi-Generation

**Model:** Qwen2.5-Coder-14B from Kaggle Model Hub  
**GPU:** T4 16 GB — 4-bit quant (~7 GB VRAM)  
**Dashboard:** Live via cloudflared public URL  
**Multi-gen:** Supervisor loop runs Gen 1 → Gen 2 → Gen 3... in one session

### Before running:
1. `Settings → Accelerator → GPU T4 x1` ✅
2. `Settings → Internet → On` ✅
3. `+ Add Input → Models → qwen2.5-coder → 14b-instruct` ✅

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────
import subprocess, torch
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                    '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip() or '❌ NOT FOUND — enable GPU T4!')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── Cell 2: Find Qwen2.5-Coder-14B model path ────────────────────────
import os, glob

candidates = glob.glob('/kaggle/input/**/config.json', recursive=True)
qwen_paths = [os.path.dirname(p) for p in candidates
              if 'qwen' in p.lower() or 'coder' in p.lower()]

if not qwen_paths:
    print('❌ Model not found. Available inputs:')
    for d in os.listdir('/kaggle/input'):
        print(f'  /kaggle/input/{d}/')
    print('\n👉 Add: + Add Input → Models → qwen2.5-coder → 14b-instruct')
    MODEL_PATH = None
else:
    MODEL_PATH = qwen_paths[0]
    print(f'✅ Model at: {MODEL_PATH}')
    print('Files:', os.listdir(MODEL_PATH)[:6])

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────
import subprocess, sys

pkgs = [
    'fastapi', 'uvicorn[standard]',
    'bitsandbytes', 'accelerate',
    'langchain-groq', 'langchain-core', 'langchain-openai',
    'langgraph', 'langchain-community', 'langchain-google-genai',
    'cerebras-cloud-sdk', 'openai',
]
print('Installing (1-2 min)...')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs,
                   capture_output=True, text=True)
print('✅ Done' if r.returncode == 0 else f'❌ {r.stderr[-200:]}')

In [ ]:
# ── Cell 4: Write LLM shim (Ollama-compatible API over transformers) ──
SHIM = '''
import json, logging, time, uuid, os
from typing import List, Optional
import torch, uvicorn
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
log = logging.getLogger("shim")
app = FastAPI()
_model = _tokenizer = None
_model_name = os.environ.get("SHIM_MODEL_NAME", "qwen2.5-coder:14b")

def load_model():
    global _model, _tokenizer
    path = os.environ["SHIM_MODEL_PATH"]
    log.info(f"Loading tokenizer from {path}")
    _tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
    log.info("Loading model 4-bit...")
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                               bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
    _model = AutoModelForCausalLM.from_pretrained(
        path, quantization_config=quant, device_map="cuda",
        trust_remote_code=True, torch_dtype=torch.float16)
    _model.eval()
    log.info(f"Model ready — {torch.cuda.memory_allocated()/1e9:.1f} GB VRAM")

class Msg(BaseModel):
    role: str
    content: str

class ChatReq(BaseModel):
    model: str = "qwen2.5-coder:14b"
    messages: List[Msg]
    max_tokens: Optional[int] = 4096
    temperature: Optional[float] = 0.2
    stream: Optional[bool] = False

@app.get("/api/tags")
def tags(): return {"models": [{"name": _model_name, "size": 9_000_000_000}]}

@app.get("/health")
def health(): return {"status": "ok", "loaded": _model is not None}

@app.post("/v1/chat/completions")
def chat(req: ChatReq):
    if _model is None: return JSONResponse({"error": "not loaded"}, status_code=503)
    msgs = [{"role": m.role, "content": m.content} for m in req.messages]
    text = _tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = _tokenizer(text, return_tensors="pt").to(_model.device)
    plen = inputs["input_ids"].shape[1]
    t0 = time.time()
    with torch.no_grad():
        out = _model.generate(**inputs, max_new_tokens=req.max_tokens or 4096,
            temperature=max(req.temperature or 0.2, 1e-6),
            do_sample=(req.temperature or 0.2) > 0.01,
            pad_token_id=_tokenizer.eos_token_id, eos_token_id=_tokenizer.eos_token_id)
    elapsed = time.time() - t0
    new_ids = out[0][plen:]
    resp = _tokenizer.decode(new_ids, skip_special_tokens=True)
    n = len(new_ids)
    log.info(f"{n} tokens {elapsed:.1f}s ({n/elapsed:.1f} tok/s)")
    return {"id": f"chatcmpl-{uuid.uuid4().hex[:8]}", "object": "chat.completion",
            "created": int(time.time()), "model": _model_name,
            "choices": [{"index": 0, "message": {"role": "assistant", "content": resp}, "finish_reason": "stop"}],
            "usage": {"prompt_tokens": plen, "completion_tokens": n, "total_tokens": plen+n}}

load_model()
uvicorn.run(app, host="0.0.0.0", port=11434, log_level="warning")
'''
with open('/kaggle/working/llm_shim.py', 'w') as f:
    f.write(SHIM)
print('✅ Shim written')

In [ ]:
# ── Cell 5: Start LLM shim (loads 14B model into GPU) ────────────────
import subprocess, sys, os, time, urllib.request, json, threading

assert MODEL_PATH, "Run Cell 2 first!"
env = os.environ.copy()
env['SHIM_MODEL_PATH'] = MODEL_PATH
env['SHIM_MODEL_NAME'] = 'qwen2.5-coder:14b'

print(f'Loading model from {MODEL_PATH}...')
print('⏳ Takes ~60-90s — watch for "Model ready"...')

shim_proc = subprocess.Popen(
    [sys.executable, '/kaggle/working/llm_shim.py'], env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

ready = threading.Event()
def _read():
    for line in shim_proc.stdout:
        print('[shim]', line.rstrip())
        if 'Model ready' in line: ready.set()
threading.Thread(target=_read, daemon=True).start()

ready.wait(timeout=180) or time.sleep(30)
print('\n✅ Model loaded!')

for i in range(10):
    try:
        with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=5) as r:
            print(f'✅ Shim ready — {[m["name"] for m in json.load(r)["models"]]}')
            break
    except:
        print(f'  waiting... ({i+1}/10)'); time.sleep(5)

In [ ]:
# ── Cell 6: Smoke test ────────────────────────────────────────────────
import urllib.request, json, time

t0 = time.time()
payload = json.dumps({'model': 'qwen2.5-coder:14b',
    'messages': [{'role': 'user', 'content': 'Python one-liner to reverse a string. One line only.'}],
    'max_tokens': 60, 'temperature': 0.1}).encode()
req = urllib.request.Request('http://localhost:11434/v1/chat/completions',
    data=payload, headers={'Content-Type': 'application/json'}, method='POST')
with urllib.request.urlopen(req, timeout=120) as r:
    resp = json.load(r)
elapsed = time.time() - t0
n = resp['usage']['completion_tokens']
print(f'Response: {resp["choices"][0]["message"]["content"].strip()}')
print(f'Speed: {n/elapsed:.1f} tok/s  {"✅ GPU" if n/elapsed > 10 else "⚠️ check GPU"}')

In [ ]:
# ── Cell 7: Clone evolution code ──────────────────────────────────────
import subprocess, os

REPO   = 'https://github.com/balaji33k/SelfReplicatingAgent.git'
BRANCH = 'fresh-main'
DEST   = '/kaggle/working/SelfReplicatingAgent'

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    REPO = REPO.replace('https://', f'https://{token}@')
    print('Using GitHub token')
except Exception:
    print('No GITHUB_TOKEN — trying public')

if os.path.exists(DEST):
    subprocess.run(['git', '-C', DEST, 'pull'], check=True)
    print('✅ Updated')
else:
    r = subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO, DEST],
                       capture_output=True, text=True)
    print('✅ Cloned' if r.returncode == 0 else f'❌ {r.stderr}')
    if r.returncode != 0: raise RuntimeError('Clone failed')

print(f'Files: {len(os.listdir(DEST + "/generations/gen_1/"))} in gen_1/')

In [ ]:
# ── Cell 8: Configure environment ────────────────────────────────────
import os, json

DEST = '/kaggle/working/SelfReplicatingAgent'
os.environ['OLLAMA_BASE_URL'] = 'http://localhost:11434'
os.environ['OLLAMA_MODEL']    = 'qwen2.5-coder:14b'
os.environ['NO_AUTO_LAUNCH']  = '1'   # supervisor loop controls sequencing

try:
    from kaggle_secrets import UserSecretsClient
    sc = UserSecretsClient()
    # HF_TOKEN → persists all generations to HuggingFace Dataset permanently
    # GROQ_API_KEY etc. → cloud fallback only (set ALLOW_CLOUD_FALLBACK=1 to activate)
    for k in ['HF_TOKEN', 'GROQ_API_KEY', 'CEREBRAS_API_KEY', 'SAMBANOVA_API_KEY', 'GoogleAPIKey']:
        try:
            v = sc.get_secret(k)
            if v: os.environ[k] = v; print(f'✅ {k}')
        except: pass
except: pass

os.makedirs(f'{DEST}/data', exist_ok=True)
with open(f'{DEST}/data/user_config.json', 'w') as f:
    json.dump({'model': 'qwen2.5-coder:14b'}, f)

print(f'OLLAMA_BASE_URL  = {os.environ["OLLAMA_BASE_URL"]}')
print(f'OLLAMA_MODEL     = {os.environ["OLLAMA_MODEL"]}')
print(f'NO_AUTO_LAUNCH   = {os.environ["NO_AUTO_LAUNCH"]}')
if os.environ.get('HF_TOKEN'):
    print('✅ HF_TOKEN set — generations will be uploaded to HuggingFace Dataset')
else:
    print('⚠️  HF_TOKEN not set — generations saved to Kaggle Output only (30 days)')
    print('   To persist permanently: add HF_TOKEN to Kaggle Secrets')


In [ ]:
# ── Cell 9: Start Live Dashboard (cloudflared public URL) ─────────────
import subprocess, os, time, threading, re, sys

DEST = '/kaggle/working/SelfReplicatingAgent'
DASHBOARD_PORT = 7860

http_server = subprocess.Popen(
    [sys.executable, '-c', f'''
import http.server, os
os.chdir("{DEST}")
class H(http.server.SimpleHTTPRequestHandler):
    def log_message(self, *a): pass
    def end_headers(self):
        self.send_header("Cache-Control", "no-store")
        self.send_header("Access-Control-Allow-Origin", "*")
        super().end_headers()
http.server.HTTPServer(("", {DASHBOARD_PORT}), H).serve_forever()
'''],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print(f'✅ HTTP server on port {DASHBOARD_PORT}')

cf_bin = '/kaggle/working/cloudflared'
if not os.path.exists(cf_bin):
    print('Downloading cloudflared...')
    import urllib.request
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        cf_bin)
    os.chmod(cf_bin, 0o755)

tunnel_proc = subprocess.Popen(
    [cf_bin, 'tunnel', '--url', f'http://localhost:{DASHBOARD_PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
def _find_url():
    global public_url
    for line in tunnel_proc.stdout:
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if m: public_url = m.group(0); break
t = threading.Thread(target=_find_url, daemon=True)
t.start(); t.join(timeout=30)

if public_url:
    print('\n' + '='*60)
    print('🎯 OPEN THIS IN YOUR BROWSER:')
    print(f'   {public_url}/MISSION_CONTROL.html')
    print('='*60)
else:
    print('⚠️ Tunnel URL not found')

In [ ]:
# ── Cell 10: Multi-Generation Supervisor Loop ─────────────────────────
# Runs Gen 1 → Gen 2 → Gen 3 ... in one Kaggle session.
# Each gen runs to completion, then the next spawned gen is picked up.
# Stop conditions: MAX_GENS reached, TARGET_PASS_RATE hit, or no next gen spawned.

import subprocess, sys, os, time, json

DEST         = '/kaggle/working/SelfReplicatingAgent'
MAX_GENS     = 5      # safety ceiling — change as needed
START_GEN    = 1      # which gen to start from
WAIT_FOR_SPAWN = 120  # seconds to wait for spawner to create next gen dir

env = os.environ.copy()
env['NO_AUTO_LAUNCH'] = '1'   # prevent detached subprocess, we loop here

for gen_num in range(START_GEN, START_GEN + MAX_GENS):
    gen_dir  = f'{DEST}/generations/gen_{gen_num}'
    main_py  = f'{gen_dir}/main.py'

    if not os.path.exists(main_py):
        print(f'\n⚠️ gen_{gen_num}/main.py not found — stopping loop.')
        break

    print(f'\n{"="*60}')
    print(f'🧬  GENERATION {gen_num}  —  {gen_dir}')
    print(f'{"="*60}')

    proc = subprocess.Popen(
        [sys.executable, 'main.py', '--generation', str(gen_num)],
        cwd=gen_dir, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)

    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
    except KeyboardInterrupt:
        proc.kill()
        print('\n⚠️ Interrupted by user')
        break

    proc.wait()
    print(f'\nGen {gen_num} exited (code {proc.returncode})')

    # ── Read pass rate from results ───────────────────────────────────
    progress_file = f'{DEST}/data/gen_{gen_num}_progress.json'
    if os.path.exists(progress_file):
        with open(progress_file) as f:
            d = json.load(f)
        results = d.get('results', {})
        task_list = d.get('task_list', [])
        success = sum(1 for r in results.values() if r['status'] == 'success')
        skipped = sum(1 for r in results.values() if r.get('error_type') == 'SkippedUnsolvable')
        attempted = len(task_list) - skipped
        pass_rate = success / attempted * 100 if attempted else 0
        print(f'Gen {gen_num} pass rate: {pass_rate:.0f}% ({success}/{attempted})')

        if pass_rate >= 95:
            print('🎉 Target pass rate (95%) reached — stopping.')
            break

    # ── Wait for next generation to be spawned ────────────────────────
    next_dir = f'{DEST}/generations/gen_{gen_num + 1}'
    next_main = f'{next_dir}/main.py'
    print(f'Waiting for gen_{gen_num+1} to be spawned (up to {WAIT_FOR_SPAWN}s)...')
    for _ in range(WAIT_FOR_SPAWN):
        if os.path.exists(next_main):
            print(f'✅ gen_{gen_num+1} ready — continuing...')
            break
        time.sleep(1)
    else:
        print(f'⏰ gen_{gen_num+1} not spawned after {WAIT_FOR_SPAWN}s — stopping loop.')
        break

print('\n🏁 Supervisor loop finished.')

In [ ]:
# ── Cell 11: Save all generations to Kaggle output + HuggingFace ─────
import shutil, os, glob, json

DEST = '/kaggle/working/SelfReplicatingAgent'
OUT  = '/kaggle/working/evolution_results'
os.makedirs(OUT, exist_ok=True)

print('=== Saving evolution artifacts ===\n')

# Save data files (heartbeat, pass rates, lineage)
for f in glob.glob(f'{DEST}/data/*.json'):
    shutil.copy(f, f'{OUT}/{os.path.basename(f)}')
    print(f'  ✅ data/{os.path.basename(f)}')

# Save each generation directory (code + evolution_artifacts + dna.json)
for gen_dir in sorted(glob.glob(f'{DEST}/generations/gen_*')):
    name = os.path.basename(gen_dir)
    dst  = f'{OUT}/{name}'
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(gen_dir, dst, ignore=shutil.ignore_patterns('__pycache__', 'sandbox'))
    
    # Show what was saved for this generation
    artifacts = f'{dst}/evolution_artifacts'
    has_artifacts = os.path.exists(artifacts)
    code_files    = len(glob.glob(f'{dst}/*.py'))
    has_dna       = os.path.exists(f'{dst}/dna.json')
    has_manifest  = os.path.exists(f'{dst}/manifest.json')
    
    print(f'  ✅ {name}/')
    print(f'     {code_files} code files | dna.json={has_dna} | manifest={has_manifest} | evolution_artifacts={has_artifacts}')
    if has_artifacts:
        for af in glob.glob(f'{artifacts}/*.md') + glob.glob(f'{artifacts}/*.json'):
            print(f'       📄 {os.path.basename(af)}')

print(f'\n📦 All saved → Kaggle Output tab (download as zip)')
print(f'   Path: /kaggle/working/evolution_results/')

# Summary table
print('\n=== Generation Summary ===')
for gen_dir in sorted(glob.glob(f'{OUT}/gen_*')):
    manifest = f'{gen_dir}/manifest.json'
    dna      = f'{gen_dir}/dna.json'
    name     = os.path.basename(gen_dir)
    if os.path.exists(manifest):
        m = json.load(open(manifest))
        pr = m.get('PARENT_PASS_RATE') or m.get('pass_rate', '?')
        print(f'  {name}: parent_pass_rate={pr}')
    elif os.path.exists(dna):
        d = json.load(open(dna))
        print(f'  {name}: parent_pass_rate={d.get("parent_pass_rate", "?")}')
